<a href="https://colab.research.google.com/github/foner21/AISE4010_Section4_Group5/blob/main/AISE4010_Section4Group5_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AISE 4010 — Room Occupancy Estimation
## Labs 1–3 and Stage 1 (Lab 6): Professor-style version

This notebook follows the **short, numbered, teaching-oriented cells** in the uploaded *Tutorial 1 Statistical Forecasting* and *Lab 3 Section 4 Solution*. It adapts the professor's temperature **regression** examples to a room-occupancy **multiclass classification** task.

**Course links:** Lecture 1 (time split and forecasting windows); Lecture 2 (audit, missing data); Lecture 3 / Lab 3 (windowing, MLP, early stopping and grid search); Lecture 4 (1D CNN); Lecture 5 (RNN, later stage).

**Problem:** predict `Room_Occupancy_Count` **2 observations ahead** from the **previous 20 observations** of sensor readings and historical occupancy. Sampling is approximately 30–31 seconds (verified in your diagnostic; not strictly 30 seconds).

**Evaluation:** chronological 60/20/20 split, training-only imputation/scaling, validation-only tuning. Test remains sealed until the professor/TA authorizes final evaluation.

**Important:** All outputs are generated when this notebook runs. Replace the group number/name, interpret graphs, cite the dataset and disclose assistance as required by your course.


## 0. Dependencies, reproducibility, data download
The authoritative project artifact must be the notebook. Use relative paths when running locally. In Colab the working directory is `/content`; all paths below are relative to it. Python packages are installed in the active notebook environment.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn tensorflow joblib

In [ ]:
import os, io, zipfile, urllib.request, warnings, time, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED)
EXPECTED_FREQ = pd.Timedelta(seconds=30)
LOOKBACK = 20
HORIZON = 2
TARGET = 'Room_Occupancy_Count'
RUN_DEEP_MODELS = True  # change to False to complete Labs 1-2 faster
RUN_GRID_SEARCH = True  # grid-search one of the Stage 1 neural models
RUN_FINAL_TEST = False  # do not change without TA permission/final protocol
DATA_DIR = Path('data'); DATA_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
print('Working directory:', Path.cwd())

In [ ]:
# Download official UCI archive, extract the occupancy CSV, otherwise prompt for upload.
csv_path = DATA_DIR / 'Occupancy_Estimation.csv'
uci_archive = 'https://archive.ics.uci.edu/static/public/864/room+occupancy+estimation.zip'
if not csv_path.exists():
    try:
        request = urllib.request.Request(uci_archive, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(request, timeout=45) as response:
            archive_bytes = response.read()
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            names = [name for name in archive.namelist() if name.lower().endswith('.csv') and not name.startswith('__MACOSX')]
            print('CSV members in archive:', names)
            candidates = [name for name in names if 'occupancy' in name.lower()]
            if not candidates: raise ValueError('Expected occupancy CSV not found in downloaded zip')
            with archive.open(candidates[0]) as src:
                csv_path.write_bytes(src.read())
        print('Dataset saved to:', csv_path)
    except Exception as exc:
        print('Automatic download failed:', repr(exc))
        print('Download the CSV from the UCI dataset page and upload it in the prompt.')
        try:
            from google.colab import files
            uploaded = files.upload()
            csv_candidates = [(name, contents) for name, contents in uploaded.items() if name.lower().endswith('.csv')]
            if not csv_candidates: raise RuntimeError('Please upload a CSV file, then rerun this cell.')
            csv_path.write_bytes(csv_candidates[0][1])
        except ImportError:
            raise RuntimeError('Place Occupancy_Estimation.csv in the local data/ folder.') from exc
assert csv_path.exists(), 'Dataset file still missing'
print('Bytes:', csv_path.stat().st_size)

## 1. Problem definition (Lab 1)
**Objective:** predict the **number of people in the room** (`Room_Occupancy_Count`) using historical sensor measurements, historical occupancy labels and known-in-advance calendar variables. Target unit: *people* (integer class), so this is multi-class classification rather than continuous regression.

**Native frequency:** expected 30 seconds, verified below. **Resampling rule:** no resampling in the initial pipeline to preserve short occupancy transitions. **Lookback:** 20 records (10 minutes). **Forecast horizon:** 2 records (1 minute after the latest input measurement).

**Operational application:** anticipate occupancy for building HVAC/ventilation and lighting management. **Leakage policy:** future sensor measurements and future true occupancy labels cannot appear in model features; preprocessing is fitted on training data only. Long timestamp gaps are not bridged in temporal windows.

**Limitations:** extremely short observation period and potential class shifts across chronological partitions. Any candidate CO₂ slope using future observations is excluded; a strictly past-only derived slope can be added in future experiments.

In [ ]:
raw = pd.read_csv(csv_path)
raw.columns = raw.columns.str.strip()
print('Shape:', raw.shape)
print('Columns:', raw.columns.tolist())
display(raw.head(8))
print('\nDtypes:'); display(raw.dtypes.to_frame('dtype'))
required = {'Date', 'Time', TARGET}
assert required.issubset(raw.columns), f'Missing expected columns: {required - set(raw.columns)}'
raw['timestamp'] = pd.to_datetime(raw['Date'].astype(str) + ' ' + raw['Time'].astype(str), errors='coerce')
invalid_ts = int(raw['timestamp'].isna().sum())
if invalid_ts: print(f'WARNING: {invalid_ts} unparseable timestamps; these rows will be excluded after audit.')
df = raw.dropna(subset=['timestamp']).sort_values('timestamp').set_index('timestamp')
assert df.index.is_monotonic_increasing

## 2. Timestamp audit — Lecture 1 and Lecture 2

Count 30 s and 31 s intervals separately. An interval exceeding 35 s is considered a **break for the modelling windows**, whereas the audit also records all intervals over nominal 30 s. This distinction prevents ordinary timestamp jitter from being mistaken for a major data gap.


In [ ]:
diffs = df.index.to_series().diff()
gap_mask = diffs > EXPECTED_FREQ
gaps = diffs.loc[gap_mask]
dup_count = int(df.index.duplicated().sum())
na_cols = df.isna().sum()
audit = pd.DataFrame([
    ['Source', 'UCI Room Occupancy Estimation, dataset #15'],
    ['Target', TARGET], ['Target unit', 'people (class label)'],
    ['Expected native frequency', '30 seconds — verify with observed intervals'],
    ['Observed most common interval', str(diffs.mode().iloc[0]) if not diffs.mode().empty else 'N/A'],
    ['Coverage start', str(df.index.min())], ['Coverage end', str(df.index.max())],
    ['Rows', len(df)], ['Unparseable date/time rows', invalid_ts],
    ['Duplicate timestamps', dup_count], ['Intervals exceeding 30 s', len(gaps)],
    ['Largest gap', str(gaps.max()) if len(gaps) else 'None'],
    ['Missing table cells', int(df.isna().sum().sum())],
], columns=['Field', 'Result'])
display(audit)
print('TOP OBSERVED TIME INTERVALS:'); display(diffs.value_counts().head(12).rename_axis('interval').to_frame('count'))
print('LARGEST GAPS:'); display(gaps.nlargest(10).to_frame('gap_duration'))
na_summary = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': (100 * df.isna().mean()).round(3)})
print('MISSINGNESS:'); display(na_summary)
print('DUPLICATES:', dup_count)

In [ ]:
# Duplicate times are not averaged silently. We keep first row only after the audit,
# and report the exact count. Edit policy if true duplicate rows require special handling.
if dup_count:
    print(f'Cleaning note: dropping {dup_count} repeated timestamps (keeping first).')
    df = df.loc[~df.index.duplicated(keep='first')].copy()
else:
    df = df.copy()

# Convert numeric sensor/label fields while keeping Date/Time out of the feature matrix.
for col in df.columns.difference(['Date', 'Time']):
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Target missing rows:', df[TARGET].isna().sum())
# Target rows cannot be reliably imputed in supervised forecasting.
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)
print('Post-audit rows:', len(df))

## 3. Exploratory data analysis (Lab 1)
**Figure 1:** target class distribution. **Figure 2:** observed occupancy across the measured interval. **Figure 3:** CO₂ versus time, coloured by occupant count. **Figure 4:** per-sensor distribution by class. Add a sentence below each figure after actually inspecting results.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
counts = df[TARGET].value_counts().sort_index()
counts.plot.bar(ax=ax, color='#4278a8')
ax.set(xlabel='Number of people', ylabel='Observed rows', title='Figure 1. Class distribution')
for i, value in enumerate(counts): ax.text(i, value, str(value), ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()
display(pd.DataFrame({'count':counts, 'percent':(100*counts/counts.sum()).round(2)}))

**Figure 1 interpretation:** *Fill after running: Which class is most common? Is a majority-class baseline likely to look deceptively strong on accuracy?*

In [ ]:
fig, ax = plt.subplots(figsize=(14,4))
ax.step(df.index, df[TARGET], where='post', linewidth=0.7)
ax.set(title='Figure 2. Room occupancy over time', ylabel='People', xlabel='Timestamp')
plt.tight_layout(); plt.show()

**Figure 2 interpretation:** *Fill after running: Are there long empty-room periods, rapid transitions, or markedly different day-level patterns?*

In [ ]:
co2_candidates = [c for c in df.columns if c.lower() in ['s5_co2', 'co2']]
if co2_candidates:
    co2_col = co2_candidates[0]
    fig, ax = plt.subplots(figsize=(14,4))
    classes = sorted(df[TARGET].unique())
    for label in classes:
        part = df[df[TARGET] == label]
        ax.scatter(part.index, part[co2_col], s=2.5, alpha=0.55, label=f'{label} people')
    ax.set(title='Figure 3. CO₂ measurements coloured by room occupancy', xlabel='Timestamp', ylabel=co2_col)
    ax.legend(markerscale=3, ncol=min(5,len(classes)))
    plt.tight_layout(); plt.show()
else: print('CO₂ field not found. Available columns:',df.columns.tolist())

**Figure 3 interpretation:** *Fill after running: Does CO₂ increase with occupancy? Is there a lag in the response? The plot alone does not establish causality.*

In [ ]:
sensor_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
# CO2 slope may be defined using future sensor points; retain for audit only, never as model input.
plot_cols = [c for c in sensor_cols if 'slope' not in c.lower()]
if plot_cols:
    ncols = 3
    nrows = (len(plot_cols)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows,ncols,figsize=(15,3.2*nrows),squeeze=False)
    for ax, sensor in zip(axes.flat,plot_cols):
        sns.boxplot(data=df,x=TARGET,y=sensor,ax=ax,showfliers=False)
        ax.set_title(sensor); ax.set_xlabel('People')
    for ax in axes.flat[len(plot_cols):]: ax.axis('off')
    fig.suptitle('Figure 4. Sensor measurements by occupancy count', y=1.01)
    plt.tight_layout(); plt.show()

**Figure 4 interpretation:** *Fill after running: Identify any sensors whose class distributions differ and those which are poor class separators.*

## 4. Chronological split and gap diagnostics (Lab 2)
No shuffling. We split before fitting preprocessors or constructing windows. Check class coverage: the brief period of available data can yield splits without all occupant counts.

In [ ]:
n = len(df); train_end = int(n*0.6); val_end = int(n*0.8)
train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()
all_classes = sorted(df[TARGET].unique())
summary=[]
for name, part in [('train',train_df),('validation',val_df),('test (sealed)',test_df)]:
    summary.append({'split':name,'rows':len(part),'start':part.index.min(),'end':part.index.max(),
                    'classes':str(part[TARGET].value_counts().sort_index().to_dict()),
                    'missing_classes':str(sorted(set(all_classes)-set(part[TARGET].unique())))})
display(pd.DataFrame(summary))
fig, ax=plt.subplots(figsize=(13,2.5))
for name, part, color in [('Train',train_df,'#43866e'),('Validation',val_df,'#d8a645'),('Test (sealed)',test_df,'#688fcb')]:
    ax.axvspan(part.index[0],part.index[-1],color=color,alpha=.65,label=name)
ax.set_yticks([]);ax.set_xlabel('Timestamp');ax.set_title('Figure 5. Chronological train/validation/test split')
ax.legend(loc='upper center',ncol=3);plt.tight_layout();plt.show()

**Split interpretation:** *Fill after running. If any occupancy class is absent from training, the supervised model cannot learn it. Do not randomly shuffle just to force class coverage. Ask the TA about acceptable evaluation/collection adjustments.*

**CO₂ slope leakage decision:** The supplied `S5_CO2_Slope` may have been calculated from measurements whose temporal support is not documented in the course handout. For safety, exclude **every column containing `slope`** unless its past-only definition can be established. If recreating, use `(CO2[t] - CO2[t-1]) / 30s`, never a centered difference or future CO₂.

## 5. Train-only preprocessing (Lab 2)
We retain the native frequency. Calendar features represent known-in-advance values for the **input timestamps**. A median imputer and standard scaler are fitted on training *only* and applied unchanged to validation/test.

A warning: generating imputed/scaled frames is not the same as making forecasting windows; the windows below will reject any missing or irregular timestamp jump across their entire lookback-to-target interval.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

safe_sensor_cols = [c for c in sensor_cols if 'slope' not in c.lower()]
assert TARGET not in safe_sensor_cols

def add_calendar(part):
    part = part.copy()
    clock = part.index.hour + part.index.minute/60 + part.index.second/3600
    day = part.index.dayofweek
    part['hour_sin'] = np.sin(2*np.pi*clock/24)
    part['hour_cos'] = np.cos(2*np.pi*clock/24)
    part['dow_sin'] = np.sin(2*np.pi*day/7)
    part['dow_cos'] = np.cos(2*np.pi*day/7)
    return part

train_df = add_calendar(train_df); val_df = add_calendar(val_df); test_df = add_calendar(test_df)
calendar_cols=['hour_sin','hour_cos','dow_sin','dow_cos']
feature_cols = safe_sensor_cols + calendar_cols
imputer = SimpleImputer(strategy='median', keep_empty_features=True)
scaler = StandardScaler()
X_train_rows = scaler.fit_transform(imputer.fit_transform(train_df[feature_cols]))
X_val_rows   = scaler.transform(imputer.transform(val_df[feature_cols]))
# Sealed: transform test features if needed for final stage, but do not score them.
print('Number of lagged sensor+calendar features:',len(feature_cols))
print('Feature names:',feature_cols)
print('Train feature matrix shape:',X_train_rows.shape)

## 6. Sliding-window / lag method — Lecture 3, pages 23–27

The professor's Lab 3 function uses a simple loop and appends arrays to `X` and `y`. The same pattern appears below, extended to (1) multivariate sensor inputs, (2) future occupancy labels and (3) **gap checks** needed for this dataset.

- `LOOKBACK=20`: use 20 prior rows.
- `HORIZON=2`: predict occupancy 2 observations after the last input row.
- `MAX_INTERVAL=35 s`: 30/31 s steps qualify; 61 s steps and long recording breaks do not.
- The count from the last input observation is also the **persistence** baseline.

We use a *direct* horizon-2 target, not two iterated one-step forecasts. The forecast lead is approximately 1 minute, because sampling is not perfectly regular.


In [ ]:
# B3. Create supervised samples (Lab 3 style: X, y = [], [])
# Do not build windows across split boundaries or missing time spans.
LOOKBACK = 20
HORIZON = 2
MAX_INTERVAL = pd.Timedelta(seconds=35)

def make_windows(frame, scaled_values, lookback=LOOKBACK, horizon=HORIZON):
    X, y = [], []
    persistence, timestamps = [], []
    counts = frame[TARGET].to_numpy()

    for i in range(lookback, len(frame) - horizon + 1):
        start = i - lookback
        target_pos = i + horizon - 1
        segment = frame.index[start:target_pos + 1]
        intervals = segment.to_series().diff().dropna()

        # Only real, near-contiguous observations may form a sequence.
        if not ((intervals > pd.Timedelta(0)) &
                (intervals <= MAX_INTERVAL)).all():
            continue

        past_sensors = scaled_values[start:i]
        past_counts = counts[start:i].reshape(-1, 1)
        X.append(np.column_stack([past_sensors, past_counts]))
        y.append(int(counts[target_pos]))
        persistence.append(int(counts[i - 1]))
        timestamps.append(frame.index[target_pos])

    if len(X) == 0:
        return (np.empty((0, lookback, scaled_values.shape[1] + 1), dtype='float32'),
                np.array([], dtype=int), np.array([], dtype=int), pd.DatetimeIndex([]))

    return (np.array(X, dtype='float32'), np.array(y, dtype=int),
            np.array(persistence, dtype=int), pd.DatetimeIndex(timestamps))

X_train, y_train, persist_train, t_train = make_windows(train_df, X_train_rows)
X_val, y_val, persist_val, t_val = make_windows(val_df, X_val_rows)

print('X_train shape:', X_train.shape, 'y_train shape:', y_train.shape)
print('X_val shape:  ', X_val.shape, 'y_val shape:  ', y_val.shape)
print('Training windows:', len(y_train))
print('Validation windows:', len(y_val))
print('Skipped windows (train):', max(0, len(train_df)-LOOKBACK-HORIZON+1)-len(y_train))
print('Skipped windows (val):', max(0, len(val_df)-LOOKBACK-HORIZON+1)-len(y_val))

assert len(y_train) > 0 and len(y_val) > 0, (
    'No valid windows. Check the timestamp counts and MAX_INTERVAL (do not ignore large gaps).')
assert t_train.max() < t_val.min(), 'Train/validation target overlap detected'
print('First training X window shape:', X_train[0].shape)
print('Class counts in training windows:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Class counts in validation windows:', dict(zip(*np.unique(y_val, return_counts=True))))


## 7. Traditional baselines (Lab 2)
- Majority class: always predicts most common **training-window** target label.
- Persistence: predicts the occupancy at the most recent **input** timestamp.
- Random forest: simple classical model using **exactly the same lagged flattened input windows** as the neural models.

Metrics are computed against the *validation* labels only. Macro-F1 and balanced accuracy are preferred when classes are imbalanced.

In [ ]:
from sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,confusion_matrix,ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from collections import Counter

results=[]
def evaluate_validation(name,pred,fit_seconds=0.0,params=0):
    row={'Model':name,'Accuracy':accuracy_score(y_val,pred),
         'Macro F1':f1_score(y_val,pred,labels=all_classes,average='macro',zero_division=0),
         'Balanced accuracy':balanced_accuracy_score(y_val,pred),
         'Parameters / trees':params,'Train seconds':fit_seconds}
    results.append(row)
    print('\n',name);display(pd.DataFrame([row]).round(4))
    cm=confusion_matrix(y_val,pred,labels=all_classes)
    fig,ax=plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm,display_labels=all_classes).plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'Validation confusion matrix — {name}')
    plt.tight_layout();plt.show()
    return row

majority_label=Counter(y_train).most_common(1)[0][0]
majority_val=np.full(len(y_val),majority_label,dtype=int)
evaluate_validation('Majority',majority_val)
evaluate_validation('Persistence',persist_val)

start=time.perf_counter()
forest=RandomForestClassifier(n_estimators=150,max_depth=12,min_samples_leaf=3,
                             random_state=SEED,n_jobs=-1)
forest.fit(X_train.reshape(len(X_train),-1),y_train)
forest_time=time.perf_counter()-start
rf_val=forest.predict(X_val.reshape(len(X_val),-1))
evaluate_validation('Random forest',rf_val,forest_time,params=forest.n_estimators)

## 8. MLP and CNN — Lecture 3 + Lab 3 + Lecture 4

Use the professor's `Sequential([Input(...), Dense(...), ...])` pattern and `model.compile(optimizer=Adam(...), loss=..., metrics=...)`. For multiclass occupancy classification we use **softmax + sparse categorical cross entropy**, rather than a single linear output + MSE used in the temperature regression lab.

- **MLP:** `Flatten` the time window, then `Dense` + ReLU.
- **CNN:** `Conv1D` slides filters along time, then pools local responses.
- **Both:** same X and y windows, 20 epochs maximum, early stopping and validation-only selection.


In [ ]:
# B5. MLP and CNN: Sequential API (adapted from Lab 3)
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.layers import GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(SEED)
classes = np.array(all_classes)
class_to_idx = {value: i for i, value in enumerate(classes)}
y_train_idx = np.array([class_to_idx[value] for value in y_train])
y_val_idx = np.array([class_to_idx[value] for value in y_val])
n_classes = len(classes)

EPOCHS = 20
BATCH_SIZE = 128

def build_mlp(units=32, lr=0.001):
    model = Sequential([
        Input(shape=X_train.shape[1:]),
        Flatten(),
        Dense(units, activation='relu'),
        Dropout(0.2),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_cnn(filters=32, kernel_size=3, lr=0.001):
    model = Sequential([
        Input(shape=X_train.shape[1:]),
        Conv1D(filters=filters, kernel_size=kernel_size,
               padding='causal', activation='relu'),
        MaxPooling1D(pool_size=2),
        Conv1D(filters=filters, kernel_size=kernel_size,
               padding='causal', activation='relu'),
        GlobalAveragePooling1D(),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def fit_and_predict(model, epochs=EPOCHS):
    early_stop = EarlyStopping(monitor='val_loss', patience=4,
                               restore_best_weights=True)
    start = time.perf_counter()
    history = model.fit(X_train, y_train_idx,
                        validation_data=(X_val, y_val_idx),
                        epochs=epochs, batch_size=BATCH_SIZE,
                        shuffle=False, callbacks=[early_stop], verbose=1)
    elapsed = time.perf_counter() - start
    prediction = model.predict(X_val, verbose=0).argmax(axis=1)
    prediction = classes[prediction]
    return history, prediction, elapsed

models = {}
histories = {}
if RUN_DEEP_MODELS:
    for model_name, builder in [('MLP', build_mlp), ('CNN', build_cnn)]:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        model = builder()
        model.summary()
        history, pred, seconds = fit_and_predict(model)
        models[model_name] = model
        histories[model_name] = history
        evaluate_validation(model_name, pred, seconds, model.count_params())

        plt.figure(figsize=(8, 4))
        plt.plot(history.history['loss'], label='Train loss')
        plt.plot(history.history['val_loss'], label='Validation loss')
        plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss')
        plt.title(model_name + ' learning curves')
        plt.legend(); plt.show()
else:
    print('Neural models skipped. Change RUN_DEEP_MODELS to True to run.')


## 9. Small grid search — Lab 3 Section 4 style

The professor's grid-search example uses `itertools.product`, a concise candidate table, and selects hyperparameters from **validation** only. We apply the same pattern to the CNN, using filters `[16, 32]` and learning rates `[0.01, 0.001]` (four trials). Our primary objective is **macro-F1**, appropriate for unequal occupancy class frequencies.


In [ ]:
# B6. Small grid search — itertools.product (Lab 3 style)
from itertools import product

grid = {'filters': [16, 32], 'lr': [0.01, 0.001]}
grid_results = []
best_grid_model = None
best_grid_score = -1
best_grid_config = None

if RUN_DEEP_MODELS and RUN_GRID_SEARCH:
    for filters, lr in product(grid['filters'], grid['lr']):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        model = build_cnn(filters=filters, lr=lr)
        history, pred, elapsed = fit_and_predict(model, epochs=15)
        macro_f1 = f1_score(y_val, pred, labels=all_classes,
                            average='macro', zero_division=0)
        grid_results.append({'filters': filters, 'learning_rate': lr,
                             'validation_macro_f1': macro_f1,
                             'epochs': len(history.history['loss']),
                             'parameters': model.count_params(),
                             'training_seconds': elapsed})
        if macro_f1 > best_grid_score:
            best_grid_score = macro_f1
            best_grid_config = {'filters': filters, 'lr': lr}
            best_grid_model = model

    grid_table = pd.DataFrame(grid_results).sort_values(
        'validation_macro_f1', ascending=False)
    display(grid_table.round(4))
    print('Best grid-search configuration:', best_grid_config)
    best_pred = classes[best_grid_model.predict(X_val, verbose=0).argmax(axis=1)]
    selected = grid_table.iloc[0]
    evaluate_validation('CNN (grid selected)', best_pred,
                        float(selected['training_seconds']), best_grid_model.count_params())
else:
    print('Grid search skipped by configuration.')


## 10. Stage 1 comparison, result interpretation and risks
All reported metrics below use the same validation target timestamps. Report improvement relative to majority macro-F1. A **negative improvement** means worse than majority. Class-imbalance handling and the very short coverage are unresolved risks until tested.

In [ ]:
comparison = pd.DataFrame(results).drop_duplicates(subset='Model',keep='last')
base = float(comparison.loc[comparison['Model']=='Majority','Macro F1'].iloc[0])
comparison['Macro F1 gain vs majority (%)'] = 100*(comparison['Macro F1']-base)/base if base else np.nan
comparison = comparison.sort_values('Macro F1',ascending=False)
display(comparison.round(4))
print('Best validation model:',comparison.iloc[0]['Model'])
print('UNRESOLVED RISK: class imbalance, short recording coverage, missing classes or occupancy drift by time segment.')

**Stage 1 conclusion (write after running):** Describe the highest-scoring model **on validation**, quantify its gain versus majority and persistence, and identify one **concrete failure mode** from a confusion matrix. Don't assert generalization performance based on validation alone.

**TA decision needed:** Stage 1 asks for test comparisons, but the common protocol/final stage say leave the test set untouched until the final stage. Record the TA's written or verbal guidance here, including the date.

## 11. Test set — sealed

The course project document is inconsistent: Stage 1's summary mentions test comparison, but its core protocol explicitly reserves the test set for the final stage. Ask the TA before opening it. This notebook intentionally does not compute test predictions during Stage 1.


In [ ]:
print('Test set sealed:', len(test_df), 'original rows. No test predictions were computed.')

## 12. Reproducibility, checkpoint and contribution log
**Dataset source:** UCI Room Occupancy Estimation (dataset #15). **Required reproducibility:** exact notebook, dependencies, ordered cells, Git commit, random seed, figures and saved output.

**Group work log** (update this table regularly and cite any substantial external/AI assistance under your course rules):

| Date | Group member | Work completed | Git commit hash |
|---|---|---|---|
| YYYY-MM-DD | Name | Timestamp audit and source | pending |
| YYYY-MM-DD | Name | EDA and figures | pending |
| YYYY-MM-DD | Name | Splits and leakage checks | pending |
| YYYY-MM-DD | Name | Traditional baselines | pending |
| YYYY-MM-DD | Name | MLP/CNN and HPO | pending |

**Recommended GitHub file layout:** `AISE4010_Project_GroupNN.ipynb`, `README.md`, `requirements.txt`, `data/README.md`; do not commit credentials, caches or large raw data.

**Sources:** AISE 4010 Fall 2026 Course Project handout; UCI Room Occupancy Estimation; Python libraries used in notebook. Add formal dataset publication/library citations to your final report.

### Coding correspondence to supplied teaching materials
- Tutorial 1: short cells, chronological split, displayed diagnostic plots and leakage checklist. Statistical ARIMA/SARIMA fits are not used as the principal baseline for a multiclass label.
- Lab 3 Section 4: `make_windows`, `StandardScaler`, `Sequential`, `Input`, `Dense`, `Adam`, `EarlyStopping`, `history.history`, `itertools.product`.
- Lecture 4: `Conv1D` and pooling for local time patterns.
- Lecture 5: RNN/GRU/LSTM belongs to the *next* stage, so it is documented but not substituted for the Stage 1 requirements.


In [ ]:
# Local/Colab dependency report for the README/requirements file.
import sklearn, tensorflow
print('pandas=='+pd.__version__)
print('numpy=='+np.__version__)
print('matplotlib=='+plt.matplotlib.__version__)
print('seaborn=='+sns.__version__)
print('scikit-learn=='+sklearn.__version__)
print('tensorflow=='+tensorflow.__version__)
print('Seed:',SEED,'lookback:',LOOKBACK,'horizon:',HORIZON)

### Demonstration checklist (Stage 1 — Lab 6)
- [ ] Dataset registration with TA confirmed
- [ ] Audit shows observed frequency, gaps, duplicate and missing values
- [ ] At least two informative figures with written interpretations
- [ ] Split diagram and class coverage explained
- [ ] Imputer/scaler fit only on train; windows do not bridge time gaps or splits
- [ ] Validated majority, persistence and traditional baseline results
- [ ] MLP/CNN training and learning curves
- [ ] HPO trials, selected settings and fair comparison table
- [ ] One supported conclusion, one unresolved risk
- [ ] Executed notebook committed, group log updated, each person can explain the pipeline